### Dependencias, Ingesta y Copia de Seguridad

In [48]:
import pandas as pd
import numpy as np
import ftfy
import unicodedata

# 1. Cargar el archivo crudo respetando su delimitador y codificación de origen
df_raw = pd.read_csv('../data/raw/oferta_gastronomica.csv', sep=';', encoding='latin1')

# 2. Crear una copia de trabajo independiente para no alterar nunca los datos originales
df = df_raw.copy()

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")


Dataset cargado: 2823 filas x 18 columnas


### Diagnostico de datos

In [49]:
diagnostico = pd.DataFrame({
    'Tipo_Dato': df.dtypes,
    'Nulos': df.isnull().sum(),
    'Porcentaje_Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Valores_Unicos': df.nunique()
})

diagnostico


,Tipo_Dato,Nulos,Porcentaje_Nulos,Valores_Unicos
long,str,0,0.00,2703
lat,str,0,0.00,2702
id,int64,0,0.00,2823
nombre,str,0,0.00,2554
categoria,str,116,4.11,8
cocina,str,359,12.72,52
ambientacion,str,559,19.80,57
telefono,str,520,18.42,2246
mail,str,2351,83.28,403
horario,str,2098,74.32,387


### Reparación Global de Codificación (Mojibake)

In [50]:
def reparar_texto_global(valor):
    if pd.isna(valor) or not isinstance(valor, str):
        return valor
    
    # Paso 1: Reparación heurística automática con ftfy
    texto = ftfy.fix_text(valor)
    
    # Paso 2: Reversión en cascada para residuos de doble/triple codificación Windows
    for _ in range(2):
        reparado = False
        for enc in ['latin1', 'cp1252']:
            try:
                candidato = texto.encode(enc).decode('utf-8')
                if candidato != texto:
                    texto = candidato
                    reparado = True
                    break
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
        if not reparado:
            break
            
    return texto

cols_objeto = df.select_dtypes(include=['object', 'string']).columns.tolist()
df[cols_objeto] = df[cols_objeto].map(reparar_texto_global)


### Estandarización de Textos y Normalización Unicode

In [51]:
def remover_tildes(valor):
    if pd.isna(valor) or not isinstance(valor, str):
        return valor
    return ''.join(
        c for c in unicodedata.normalize('NFD', valor)
        if unicodedata.category(c) != 'Mn'
    )

# 1. Limpieza general en todas las columnas de texto (excepto 'mail' y 'horario' que se procesan aparte)
cols_limpieza_general = [c for c in cols_objeto if c not in ['mail', 'horario']]

for col in cols_limpieza_general:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
    )

# 2. Quitar tildes en columnas de agrupación para evitar fragmentar datos
cols_sin_tildes = ['barrio', 'categoria', 'cocina', 'ambientacion']
for col in cols_sin_tildes:
    df[col] = df[col].apply(remover_tildes)


### Coordenadas Geográficas, Imputación de Nulos y Descarte

In [52]:
# 1. Convertir coordenadas de texto con coma a números decimales (float)
for coord in ['long', 'lat']:
    df[coord] = (
        df[coord]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, 'NAN': np.nan})
        .astype(float)
    )

# 2. Imputar nulos en columnas categóricas clave para no perder registros al agrupar
for col in ['categoria', 'cocina', 'ambientacion']:
    df[col] = df[col].fillna('SIN ESPECIFICAR')

# 3. Eliminar columna casi vacía (99.7% nulos) y duplicados por ID
df = df.drop(columns=['calle_cruce'], errors='ignore')
df = df.drop_duplicates(subset=['id']).reset_index(drop=True)


### Coordenadas, Imputación y eliminacion de columnas (Enfoque Punto 2 y Cruce)

In [53]:
# 1. Convertir coordenadas de texto con coma a números decimales (float)
for coord in ['long', 'lat']:
    df[coord] = (
        df[coord]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, 'NAN': np.nan})
        .astype(float)
    )

# 2. Imputar nulos en columnas categóricas clave para no perder registros al agrupar
for col in ['categoria', 'cocina', 'ambientacion']:
    df[col] = df[col].fillna('SIN ESPECIFICAR')

# 3. Eliminar duplicados por ID antes de podar columnas
df = df.drop_duplicates(subset=['id']).reset_index(drop=True)

# 4. Eliminacion de columnas con nulos masivos o sin valor analítico para el Punto 2 y cruce con Trends
columnas_a_eliminar = [
    'calle_cruce',              # 99.79% nulos (sin valor)
    'telefono',                 # Dato operativo de contacto, irrelevante para tendencias
    'codigo_postal_argentino',  # Redundante frente a barrio, comuna y codigo_postal
    'calle_nombre',             # Redundante frente a direccion_completa
    'calle_altura'              # Redundante frente a direccion_completa
]

df = df.drop(columns=columnas_a_eliminar, errors='ignore')


In [54]:
# 1. Auditoría de caracteres especiales residuales
for col in df.select_dtypes(include=['object', 'string']).columns:
    serie_valida = df[col].dropna()
    especiales = serie_valida[serie_valida.str.contains(r'[^\x00-\x7F]', regex=True)]
    if len(especiales) > 0:
        print(f"Columna [{col}]: {len(especiales)} filas con caracteres especiales -> {especiales.unique()[:3]}")

# 2. Exportar dataset maestro limpio en data/processed/ (Checkpoint Clase 13)
ruta_salida = '../data/processed/oferta_gastronomica_limpia.csv'
df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f"\nExportación exitosa en '{ruta_salida}': {df.shape[0]} filas x {df.shape[1]} columnas.")


Columna [nombre]: 68 filas con caracteres especiales -> <ArrowStringArray>
['A CORUÑA', 'AÑARO CERO', 'ANTAÑO BAR']
Length: 3, dtype: str
Columna [horario]: 222 filas con caracteres especiales -> <ArrowStringArray>
[                     'lunes a sábado de 8 a 00',
         'lunes a sábado de 9 a 16 y de 19 a 21',
 'lunes a viernes de 12 a 00 - sábado de 20 a 3']
Length: 3, dtype: str
Columna [direccion_completa]: 38 filas con caracteres especiales -> <ArrowStringArray>
['HUMBERTO 1º 1843', 'RODRIGUEZ PEÑA 687', 'CERVIÑO AV. 3496']
Length: 3, dtype: str

Exportación exitosa en '../data/processed/oferta_gastronomica_limpia.csv': 2823 filas x 13 columnas.
